# Test inference with checkpoint-200

Load the cumulative GRPO LoRA checkpoint, generate four completions for each test question, and save the submission in the exact sample-submission format.

In [ ]:
!git clone https://github.com/joshsalako/telelogs.git

In [ ]:
import os
import sys

print("Python:", sys.executable)

# Install uv using the notebook's actual Python.
# Use only one -q, not -qqq.
!{sys.executable} -m pip install -q --upgrade uv

# Make every uv command install into the notebook environment.
os.environ["UV_SYSTEM_PYTHON"] = "1"

In [ ]:
!uv pip install --upgrade \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo.git" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth.git" \
    bitsandbytes \
    "xformers==0.0.32.post2" \
    datasets \
    pandas

!uv pip install --upgrade --no-deps \
    "transformers==5.2.0" \
    "tokenizers>=0.22.0,<=0.23.0" \
    "trl==0.22.2" \
    "torchao>=0.16.0" \
    "huggingface-hub>=1.3.0"

In [ ]:
from importlib.metadata import version, PackageNotFoundError

packages = [
    "torch",
    "transformers",
    "trl",
    "unsloth",
    "unsloth_zoo",
    "tokenizers",
    "bitsandbytes",
    "xformers",
    "torchao",
    "huggingface-hub",
]

for package in packages:
    try:
        print(f"{package:25s}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:25s}: NOT INSTALLED")

## Configuration and paths

In [ ]:
import gc
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 8192
MAX_COMPLETION_LENGTH = 1024 * 2
MAX_PROMPT_LENGTH = MAX_SEQ_LENGTH - MAX_COMPLETION_LENGTH
LORA_RANK = 16
NUM_SAMPLES = 4
# Tuned for a T4 with 16 GB VRAM; reduce to 1 if CUDA runs out of memory.
BATCH_SIZE = 2
TEMPERATURE = 0.7
TOP_P = 0.95
CHECKPOINT_DIR = "checkpoint-200"
OUTPUT_CSV = "submission.csv"
PROGRESS_CSV = "submission.progress.csv"
RESUME = True  # Set to False to start a fresh prediction run.

# Set to a positive integer for a quick smoke test. Keep None for the official
# all-863-question test inference and complete submission file.
TEST_LIMIT = None

SYSTEM_PROMPT = (
    "You are a senior telecom root-cause analysis engineer. Analyze the supplied "
    "drive-test and engineering evidence carefully. Follow the candidate identifiers "
    "defined in the user prompt and finish with exactly one selected identifier "
    "enclosed in \\boxed{}."
    "Do not write anything after the boxed identifier."
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "A CUDA GPU runtime is required."
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())


def locate_path(name):
    candidates = [
        Path(name),
        Path("/content/telelogs") / name,
        Path("/root/telelogs") / name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {name}. Upload it to Colab or place it in the cloned repository."
    )


CHECKPOINT_PATH = locate_path(CHECKPOINT_DIR)
TEST_PATH = locate_path("data/test.csv")
SAMPLE_SUBMISSION_PATH = locate_path("data/SampleSubmission.csv")

ADAPTER_FILE = CHECKPOINT_PATH / "adapter_model.safetensors"
if not ADAPTER_FILE.is_file():
    raise FileNotFoundError(f"Adapter weights not found at {ADAPTER_FILE}")

print("Checkpoint:", CHECKPOINT_PATH)
print("Adapter weights:", ADAPTER_FILE)
print("Test data:", TEST_PATH)
print("Sample submission:", SAMPLE_SUBMISSION_PATH)

## Load the base model and checkpoint

In [ ]:
torch.cuda.empty_cache()
gc.collect()

# A100 Optimization: Enable TF32 for faster matrix multiplications
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Patch Qwen3.5 3D position IDs calculation for text-only inputs
try:
    import transformers.models.qwen3_5.modeling_qwen3_5 as qwen3_5_module

    _orig_compute_3d_position_ids = qwen3_5_module.Qwen3_5Model.compute_3d_position_ids

    def _is_empty(x):
        return x is None or (hasattr(x, "numel") and x.numel() == 0)

    def patched_compute_3d_position_ids(
        self, input_ids=None, image_grid_thw=None, video_grid_thw=None, **kwargs
    ):
        if _is_empty(image_grid_thw) and _is_empty(video_grid_thw):
            if hasattr(self, "rope_deltas"):
                self.rope_deltas = None
            return None
        try:
            return _orig_compute_3d_position_ids(
                self,
                input_ids=input_ids,
                image_grid_thw=image_grid_thw,
                video_grid_thw=video_grid_thw,
                **kwargs,
            )
        except Exception:
            if hasattr(self, "rope_deltas"):
                self.rope_deltas = None
            return None

    qwen3_5_module.Qwen3_5Model.compute_3d_position_ids = (
        patched_compute_3d_position_ids
    )
    print("Patched Qwen3_5Model.compute_3d_position_ids successfully")
except Exception as error:
    print("Warning: could not patch Qwen3_5 compute_3d_position_ids:", error)

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    fast_inference=False,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Base model loaded:", MODEL_NAME)

In [ ]:
from safetensors.torch import load_file as safe_load

adapter_weights = safe_load(str(ADAPTER_FILE))
model_state = model.state_dict()


def normalize_key(key):
    while True:
        changed = False
        for prefix in ["base_model.", "model.", "language_model."]:
            if key.startswith(prefix):
                key = key[len(prefix) :]
                changed = True
        if not changed:
            break
    return key.replace(".default.", ".")


adapter_norm = {normalize_key(key): value for key, value in adapter_weights.items()}

loaded = 0
for key, parameter in model_state.items():
    if "lora_" not in key:
        continue
    norm_key = normalize_key(key)
    if norm_key in adapter_norm:
        parameter.data.copy_(
            adapter_norm[norm_key].to(parameter.device, parameter.dtype)
        )
        loaded += 1

total_lora = sum(1 for key in model_state if "lora_" in key)
print(f"Loaded {loaded}/{total_lora} LoRA parameters from {CHECKPOINT_PATH}")

if loaded == 0 or loaded != total_lora:
    raise RuntimeError(
        f"Failed to load the complete checkpoint: loaded {loaded}/{total_lora} LoRA parameters."
    )

## Load and validate the test data and submission template

In [ ]:
test_frame = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

if list(test_frame.columns) != ["ID", "question"]:
    raise ValueError(
        f"test.csv must contain ID,question; got {list(test_frame.columns)}"
    )

if list(sample_submission.columns) != ["ID", "Target"]:
    raise ValueError(
        f"SampleSubmission.csv must contain ID,Target; got {list(sample_submission.columns)}"
    )

if len(test_frame) != 863:
    raise ValueError(f"Expected 863 test questions; got {len(test_frame)}")

if len(sample_submission) != 863 * NUM_SAMPLES:
    raise ValueError(
        f"Expected {863 * NUM_SAMPLES} sample-submission rows; got {len(sample_submission)}"
    )

if test_frame[["ID", "question"]].isna().any().any():
    raise ValueError("Test data contains missing IDs or questions")

if sample_submission["ID"].isna().any():
    raise ValueError("Sample submission contains missing IDs")

if test_frame["ID"].duplicated().any():
    raise ValueError("Test data contains duplicate IDs")

if sample_submission["ID"].duplicated().any():
    raise ValueError("Sample submission contains duplicate IDs")

submission_index = sample_submission[["ID"]].copy()
submission_index["question_ID"] = submission_index["ID"].str.replace(
    r"_([1-4])$", "", regex=True
)
submission_index["sample"] = submission_index["ID"].str.extract(
    r"_([1-4])$", expand=False
)

if submission_index["sample"].isna().any():
    raise ValueError("Every submission ID must end in _1, _2, _3, or _4")

submission_index["sample"] = submission_index["sample"].astype(int)

test_ids = set(test_frame["ID"])
submission_question_ids = set(submission_index["question_ID"])
if test_ids != submission_question_ids:
    raise ValueError("Test and sample-submission base IDs do not match")

submission_groups = submission_index.groupby("question_ID", sort=False)
for question_id, group in submission_groups:
    if set(group["sample"]) != {1, 2, 3, 4}:
        raise ValueError(
            f"{question_id} does not contain submission suffixes _1 through _4"
        )

test_records = [
    {
        "id": row.ID,
        "question": row.question,
    }
    for row in test_frame.itertuples(index=False)
]

print(
    f"Validated {len(test_records)} questions and {len(sample_submission)} submission rows"
)

In [ ]:
def prompt_messages(question):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]


prompt_lengths = []

for index, record in enumerate(test_records):
    try:
        token_ids = tokenizer.apply_chat_template(
            prompt_messages(record["question"]),
            add_generation_prompt=True,
            tokenize=True,
            enable_thinking=True,
        )
    except Exception as error:
        raise RuntimeError(f"Failed to tokenize test record {index}") from error

    prompt_length = len(token_ids)
    if prompt_length > MAX_PROMPT_LENGTH:
        raise ValueError(
            f"Test record {record['id']} is {prompt_length} prompt tokens; "
            f"maximum is {MAX_PROMPT_LENGTH}."
        )
    prompt_lengths.append(prompt_length)

prompt_lengths_array = np.asarray(prompt_lengths)

print("Prompt-length distribution:")
for percentile in [50, 75, 90, 95, 99, 100]:
    length = np.percentile(prompt_lengths_array, percentile)
    print(f"  {percentile:>3}th percentile: {length:.0f} tokens")

print(f"Longest test prompt: {max(prompt_lengths)} tokens")

## Generate four completions per question

In [ ]:
import os

from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)
model.eval()

text_tokenizer = getattr(
    tokenizer, "tokenizer", getattr(tokenizer, "text_tokenizer", tokenizer)
)

# Decoder-only batched generation requires left padding.
text_tokenizer.padding_side = "left"
if text_tokenizer.pad_token_id is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token

# Keep the wrapper and model generation settings synchronized.
if tokenizer is not text_tokenizer:
    tokenizer.padding_side = text_tokenizer.padding_side
    tokenizer.pad_token_id = text_tokenizer.pad_token_id
    tokenizer.eos_token_id = text_tokenizer.eos_token_id

model.generation_config.pad_token_id = text_tokenizer.pad_token_id
model.generation_config.eos_token_id = text_tokenizer.eos_token_id

# Check for the terminal boxed C1-C8 label without changing the completion.
FINAL_BOX_RE = re.compile(r"\\boxed\s*\{\s*(C[1-8])\s*\}\s*$")

inference_records = test_records if TEST_LIMIT is None else test_records[:TEST_LIMIT]

if TEST_LIMIT is not None:
    print("WARNING: TEST_LIMIT is set; this will create a partial submission.")

PROGRESS_COLUMNS = ["ID", "Target"]
submission_metadata = {
    row.ID: (row.question_ID, row.sample)
    for row in submission_index.itertuples(index=False)
}


def atomic_write_csv(frame, path):
    path = Path(path)
    temporary_path = path.with_name(f"{path.name}.tmp")
    frame.to_csv(temporary_path, index=False)
    os.replace(temporary_path, path)


def save_submission_progress(completion_lookup):
    rows = []
    for row in submission_index.itertuples(index=False):
        key = (row.question_ID, row.sample)
        if key in completion_lookup:
            rows.append({"ID": row.ID, "Target": completion_lookup[key]})
    frame = pd.DataFrame(rows, columns=PROGRESS_COLUMNS)
    atomic_write_csv(frame, PROGRESS_CSV)


completion_lookup = {}
progress_path = Path(PROGRESS_CSV)
if RESUME and progress_path.is_file():
    progress_frame = pd.read_csv(progress_path, keep_default_na=False)
    if list(progress_frame.columns) != PROGRESS_COLUMNS:
        raise ValueError(
            f"{PROGRESS_CSV} has unexpected columns: {list(progress_frame.columns)}"
        )
    if progress_frame["ID"].duplicated().any():
        raise ValueError(f"{PROGRESS_CSV} contains duplicate IDs")

    unknown_ids = set(progress_frame["ID"]) - set(submission_metadata)
    if unknown_ids:
        raise ValueError(
            f"{PROGRESS_CSV} contains {len(unknown_ids)} unknown submission IDs"
        )

    loaded_by_question = {}
    for row in progress_frame.itertuples(index=False):
        question_id, sample = submission_metadata[row.ID]
        loaded_by_question.setdefault(question_id, {})[sample] = str(row.Target)

    incomplete_questions = []
    for question_id, samples in loaded_by_question.items():
        if set(samples) == set(range(1, NUM_SAMPLES + 1)):
            for sample, completion in samples.items():
                completion_lookup[(question_id, sample)] = completion
        else:
            incomplete_questions.append(question_id)

    if incomplete_questions:
        print(
            f"Regenerating {len(incomplete_questions)} incomplete questions from "
            f"{PROGRESS_CSV}."
        )
    print(
        f"Loaded {len(completion_lookup) // NUM_SAMPLES} completed questions from "
        f"{PROGRESS_CSV}."
    )
    save_submission_progress(completion_lookup)
elif not RESUME:
    if progress_path.is_file():
        print(f"RESUME is disabled; replacing existing {PROGRESS_CSV}.")
    save_submission_progress(completion_lookup)

completed_question_ids = {question_id for question_id, _ in completion_lookup}
remaining_records = [
    record for record in inference_records if record["id"] not in completed_question_ids
]
print(
    f"Test resume status: {len(inference_records) - len(remaining_records)} "
    f"completed, {len(remaining_records)} remaining."
)

progress_bar = tqdm(
    range(0, len(remaining_records), BATCH_SIZE), desc="Test inference batches"
)
for batch_start in progress_bar:
    batch = remaining_records[batch_start : batch_start + BATCH_SIZE]
    prompts = [
        tokenizer.apply_chat_template(
            prompt_messages(record["question"]),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        for record in batch
    ]

    inputs = text_tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=False,
        add_special_tokens=False,
    ).to(model.device)

    # Every generated row includes the full left-padded input width.
    prompt_width = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            do_sample=True,
            num_return_sequences=NUM_SAMPLES,
            max_new_tokens=MAX_COMPLETION_LENGTH,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            use_cache=True,
            pad_token_id=text_tokenizer.pad_token_id,
            eos_token_id=text_tokenizer.eos_token_id,
        )

    completions = text_tokenizer.batch_decode(
        output_ids[:, prompt_width:],
        skip_special_tokens=True,
    )

    expected_completions = len(batch) * NUM_SAMPLES
    if len(completions) != expected_completions:
        raise RuntimeError(
            f"Expected {expected_completions} batch completions; got {len(completions)}"
        )

    for batch_index, record in enumerate(batch):
        completion_start = batch_index * NUM_SAMPLES
        record_completions = completions[
            completion_start : completion_start + NUM_SAMPLES
        ]

        for sample, completion in enumerate(record_completions, 1):
            completion = completion.strip()
            completion_lookup[(record["id"], sample)] = completion

    save_submission_progress(completion_lookup)
    completed_in_scope = len(inference_records) - (
        len(remaining_records) - min(batch_start + BATCH_SIZE, len(remaining_records))
    )
    progress_bar.set_postfix(
        saved=completed_in_scope,
        remaining=len(inference_records) - completed_in_scope,
    )

selected_keys = [
    (record["id"], sample)
    for record in inference_records
    for sample in range(1, NUM_SAMPLES + 1)
]
missing_keys = [key for key in selected_keys if key not in completion_lookup]
if missing_keys:
    raise RuntimeError(f"Missing {len(missing_keys)} test completions after inference")

format_checks = []
for question_id, sample in selected_keys:
    completion = completion_lookup[(question_id, sample)]
    match = FINAL_BOX_RE.search(completion)
    format_checks.append(
        {
            "ID": f"{question_id}_{sample}",
            "prediction": match.group(1) if match else None,
            "format_valid": match is not None,
        }
    )

## Build and save submission.csv

In [ ]:
evaluated_ids = {record["id"] for record in inference_records}
output_index = submission_index[
    submission_index["question_ID"].isin(evaluated_ids)
].copy()

submission = output_index[["ID"]].copy()
submission["Target"] = [
    completion_lookup[(row.question_ID, row.sample)]
    for row in output_index.itertuples(index=False)
]

expected_rows = len(inference_records) * NUM_SAMPLES
if len(submission) != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows} submission rows; got {len(submission)}"
    )

if list(submission.columns) != ["ID", "Target"]:
    raise RuntimeError(f"Unexpected submission columns: {list(submission.columns)}")

if submission["ID"].duplicated().any():
    raise RuntimeError("Generated submission contains duplicate IDs")

if submission[["ID", "Target"]].isna().any().any():
    raise RuntimeError("Generated submission contains missing IDs or completions")

if TEST_LIMIT is None:
    if len(submission) != len(sample_submission):
        raise RuntimeError(
            "Full submission row count does not match SampleSubmission.csv"
        )
    if submission["ID"].tolist() != sample_submission["ID"].tolist():
        raise RuntimeError(
            "Full submission ID order does not match SampleSubmission.csv"
        )

atomic_write_csv(submission, OUTPUT_CSV)

format_frame = pd.DataFrame(format_checks)
format_valid = int(format_frame["format_valid"].sum())
format_valid_rate = format_valid / len(format_frame) if len(format_frame) else 0.0

print(
    f"Valid output format: {format_valid_rate:.2%} ({format_valid}/{len(format_frame)})"
)
print(f"Saved {len(submission)} rows to {OUTPUT_CSV}")

display(submission.head(8))

In [ ]:
try:
    from google.colab import files

    files.download(OUTPUT_CSV)
except ImportError:
    print("Not running in Colab; submission.csv remains in the current directory.")